# CuTeDSL 04: GEMM 极限性能实现

本 notebook 逐步演示如何基于 Blackwell（tcgen05）实现 SOL（Speed Of Light，极限速度）GEMM（通用矩阵乘法）。

### 学习目标

在本教程中，你将逐步学习编写高效的 GEMM：
- 如何使用异步流水线实现高效的生产者-消费者模式
- 如何使用 CuTeDSL 实现基本的 GEMM kernel
- 如何对累加器进行子分块
- 如何使用软件流水线应用多阶段
- 如何向量化存储输出的指令



In [24]:
import torch

import cutlass
import cutlass.cute as cute
import cutlass.utils as utils
import cutlass.torch as cutlass_torch
import cutlass.pipeline as pipeline
from cutlass.cute.nvgpu import cpasync, tcgen05
import cutlass.utils.blackwell_helpers as sm100_utils
from cutlass.cute.runtime import from_dlpack

## 一、异步流水线

在 GEMM kernel 中，数据搬运（TMA 加载）和计算（MMA 指令）是异步执行的，需要流水线机制来协调。本节先独立讲解异步流水线的原理和 API，为后续 GEMM 实现打基础。

### 1.1 背景：同步通信的局限

GPU 上同一 CTA（block）内的多个 warp 通过 **shared memory（SMEM）** 通信。传统做法是用 `__syncthreads()` 做全局同步：

```mermaid
sequenceDiagram
  participant W0 as Warp0_Producer
  participant SMEM as SharedMemory
  participant W1 as Warp1_Consumer

  W0->>SMEM: 写入数据
  W0<-->W1: __syncthreads()（全 CTA 同步）
  SMEM->>W1: 读取数据
  W0<-->W1: __syncthreads()（全 CTA 同步）
```

这种方式的问题是所有 warp 都在屏障处等待，生产者写完后消费者才能读，消费者读完后生产者才能写下一轮，强制串行化，导致整体吞吐量受限于最慢的一方。


下面的代码演示了这种同步模式。Warp 0 写入 SMEM，两次 `sync_threads()` 分别保证"写完再读"和"读完再写"。

In [25]:
@cute.kernel
def synced_producer_consumer(SharedStorage: cutlass.Constexpr, res: cute.Tensor):
    # 返回当前 thread 所在的 warp 编号。thread 0~31 是 warp 0，thread 32~63 是 warp 1。
    warp_idx = cute.arch.warp_idx()
    # 告诉编译器 "同一个 warp 内所有 thread 的 warp_idx 值相同"。
    # 编译器不一定知道这一点，加上这个 hint 后，编译器可以对后续的 if warp_idx == 0 分支做优化（知道整个 warp 要么全进分支，要么全不进）。
    warp_idx = cute.arch.make_warp_uniform(warp_idx)

    # 在 SMEM 中分配一个 Float32 缓冲区，作为 warp 间的通信媒介
    smem = cutlass.utils.SmemAllocator()
    # SharedStorage 是要分配的结构体类型，定义了 SMEM 中有哪些字段、每个字段的类型和大小。
    # 64 是对齐参数（alignment），单位是字节，表示分配出来的结构体整体在 SMEM 中的起始地址必须是 64 字节的整数倍
    storage = smem.allocate(SharedStorage, 64)
    # storage.staging_buffer 是一个 SMEM 结构体字段的引用，它的类型是 MemRange（一段原始内存区域），本身没有 shape/stride 等 tensor 语义。
    # get_tensor(cute.make_layout(1)) 的作用是把这段原始内存包装成一个 CuTe tensor，赋予它 layout（这里是标量 layout (1)），之后才能用 tensor 的方式访问
    staging_smem = storage.staging_buffer.get_tensor(cute.make_layout(1))
    # SMEM 是 on-chip SRAM，每次 kernel launch 时，分配给当前 block 的 SMEM 区域里可能残留着上一个 kernel 或上一个 block 留下的脏数据，所以推荐手动清零。
    staging_smem.fill(0)
    # 确保所有 thread 都完成初始化
    cute.arch.sync_threads()

    for i in cutlass.range(cute.size(res)):
        # 生产者（warp 0）：写入数据到 SMEM
        if warp_idx == 0:
            staging_smem[0] = i * 1.0
        # 同步屏障 1：保证 warp 0 写入完成后 warp 1 才能读取
        cute.arch.sync_threads()
        # 消费者（warp 1）：从 SMEM 读取数据到输出
        if warp_idx == 1:
            res[i] = staging_smem[0]
        # 同步屏障 2：保证 warp 1 读取完成后 warp 0 才能写入下一轮
        cute.arch.sync_threads()


@cute.jit
def run_synced_producer_consumer(res: cute.Tensor):
    @cute.struct
    class SharedStorage:
        # cute.struct.Align[..., 1024] 是 结构体内部的字段对齐。
        # 它控制的是 staging_buffer 这个字段在 SharedStorage 结构体内部的起始偏移量必须是 1024 字节的整数倍。
        staging_buffer: cute.struct.Align[
            cute.struct.MemRange[cutlass.Float32, 1], 1024  # 4 字节（1 个 Float32)
        ]

    synced_producer_consumer(SharedStorage, res).launch(
        grid=(1, 1, 1), block=(64, 1, 1), smem=SharedStorage.size_in_bytes()
    )


res = torch.zeros((8,), device="cuda")
run_synced_producer_consumer(from_dlpack(res))

In [26]:
res

tensor([0., 1., 2., 3., 4., 5., 6., 7.], device='cuda:0')

### 1.2 异步流水线

从 Hopper 架构开始，CUDA 引入了异步通信原语，允许不同 warp 扮演不同角色（**warp 特化**），通过流水线松耦合协作，无需全局同步。

**核心流程（4 步循环）：**

```mermaid
sequenceDiagram
  participant W0 as Warp0_Producer
  participant SMEM as SharedMemory
  participant W1 as Warp1_Consumer

  W0->>W0: 1. acquire（等待空buffer）
  W0->>SMEM: 写入数据
  W0->>W1: 2. commit（通知数据就绪）
  W1->>SMEM: 3. wait 后读取数据
  W1->>W0: 4. release（释放buffer）
```

**与同步模式的对比：**

| 对比项 | 同步模式（syncthreads） | 异步流水线（Pipeline） |
|---|---|---|
| 同步粒度 | 全 CTA 级别 | 生产者-消费者之间 |
| 等待行为 | 所有 warp 阻塞 | 仅相关方等待 |
| 重叠执行 | 不能 | 生产者可边写边让消费者处理上一轮 |
| 缓冲区管理 | 手动两次屏障 | Pipeline 自动管理buffer |


下面用单级异步流水线（`num_stages=1`）替代同步模式，实现相同的功能：

In [27]:
@cute.kernel
def async_pipeline_kernel(res: cute.Tensor):
    warp_idx = cute.arch.warp_idx()
    warp_idx = cute.arch.make_warp_uniform(warp_idx)

    @cute.struct
    class SharedStorage:
        tma_mbar_ptr: cute.struct.MemRange[cutlass.Int64, 2]   # barrier（每个 stage 需要 2 个 Int64 来存放 tma 硬件 barrier 状态）
        staging_buffer: cute.struct.Align[
            cute.struct.MemRange[cutlass.Float32, 1], 1024     # 实际的数据缓冲区（1 个 buffer， 因为 num_stages=1）
        ]

    smem = cutlass.utils.SmemAllocator()
    storage = smem.allocate(SharedStorage, 64)
    staging_smem = storage.staging_buffer.get_tensor(cute.make_layout(1))
    staging_smem.fill(0)
    cute.arch.sync_threads()

    # 定义生产者和消费者线程组（各 1 个 warp = 32 threads）
    producer_group = cutlass.pipeline.CooperativeGroup(
        cutlass.pipeline.Agent.Thread, 32
    )
    consumer_group = cutlass.pipeline.CooperativeGroup(
        cutlass.pipeline.Agent.Thread, 32
    )

    # 创建单级异步流水线（num_stages=1：只有 1 个缓冲区 buffer）
    pipe = cutlass.pipeline.PipelineAsync.create(
        num_stages=1,
        producer_group=producer_group,
        consumer_group=consumer_group,
        barrier_storage=storage.tma_mbar_ptr.data_ptr(),
    )

    producer, consumer = pipe.make_participants()

    # === 生产者（warp 0）===
    if warp_idx == 0:
        for i in cutlass.range(cute.size(res)):
            # 步骤 1: 等待空 buffer（消费者已 release），然后占据该 buffer
            # _and_advance 表示自动把内部写指针推进到下一个 buffer（num_stages=1 时总是 0→0 轮转）
            handle = producer.acquire_and_advance()
            staging_smem[handle.index] = 1.0 * i  # 写入数据到 SMEM, handle.index 在 num_stages=1 时总是 0
            # 步骤 2: 通知消费者数据就绪
            handle.commit()
        # 确保生产者最后一次 commit 发出的信号被消费者正确接收后，生产者才退出循环。
        # 因为 commit() 只是发出信号，但信号的传递是异步的（通过 SMEM 中的 barrier），消费者还要用这个信号来读数据。
        # 如果生产者在 commit() 之后立刻退出，GPU 可能会回收该 warp 的资源或让它去做别的事情，导致 barrier 状态出现异常。
        producer.tail()  # 防止生产者在所有信号到达前退出

    # === 消费者（warp 1）===
    if warp_idx == 1:
        for i in cutlass.range(cute.size(res)):
            # 步骤 3: 等待数据就绪（生产者已 commit）
            handle = consumer.wait_and_advance()
            res[i] = staging_smem[handle.index]   # 从 SMEM 读取数据
            # 步骤 4: 释放buffer给生产者
            handle.release()                           


@cute.jit
def async_pipeline(res: cute.Tensor):
    async_pipeline_kernel(res).launch(grid=(1, 1, 1), block=(64, 1, 1))


res = torch.zeros((8,), device="cuda")
async_pipeline(from_dlpack(res))

In [28]:
res

tensor([0., 1., 2., 3., 4., 5., 6., 7.], device='cuda:0')

### 1.3 多级流水线与循环缓冲

上面的单级流水线（`num_stages=1`）只有 1 个缓冲区 buffer。生产者写完第 i 条数据后，必须等消费者读完并 release，才能写第 i+1 条。这仍然是串行的。

**多级流水线**（`num_stages=N`）使用 N 个 buffer 组成**循环缓冲区**，生产者可以领先消费者最多 N-1 步。

以 `num_stages=3`、写入 8 个数据为例，展示每一步生产者和消费者各操作哪个 buffer：

```

步骤    生产者写入          消费者读取            3 个 buffer 的状态
────   ───────────────    ───────────────    ────────────────────
 0     → buf[0] 写 0                         [W] [ ] [ ]
 1     → buf[1] 写 1                         [D] [W] [ ]
 2     → buf[2] 写 2      ← buf[0] 读 0      [R] [D] [W]
 3     → buf[0] 写 3      ← buf[1] 读 1      [W] [R] [D]
 4     → buf[1] 写 4      ← buf[2] 读 2      [D] [W] [R]
 5     → buf[2] 写 5      ← buf[0] 读 3      [R] [D] [W]
 6     → buf[0] 写 6      ← buf[1] 读 4      [W] [R] [D]
 7     → buf[1] 写 7      ← buf[2] 读 5      [D] [W] [R]
 8                        ← buf[0] 读 6      [R] [D] [D]
 9                        ← buf[1] 读 7      [ ] [R] [D]

W = 正在写入   D = 数据就绪   R = 正在读取   [ ] = 空闲


- buffer 编号按 handle.index = i % num_stages 轮转（0→1→2→0→1→2→...）
- 生产者最多领先消费者 num_stages - 1 = 2 步（步骤 0~1 连续写 2 个后就要等）
- 3 个 buffer 全满时，生产者的 acquire 会阻塞，直到消费者 release 腾出空位
```


In [29]:
@cute.kernel
def async_pipeline_staged_kernel(
    SharedStorage: cutlass.Constexpr, res: cute.Tensor, staging: cute.Tensor
):
    # staging tensor 的大小决定了流水线级数（如传入 3 个元素 → 3 级流水线）
    stages = cute.size(staging)

    warp_idx = cute.arch.warp_idx()
    warp_idx = cute.arch.make_warp_uniform(warp_idx)

    smem = cutlass.utils.SmemAllocator()
    storage = smem.allocate(SharedStorage, 64)
    # SMEM 缓冲区大小 = stages，每个buffer存 1 个 Float32
    staging_smem = storage.staging_buffer.get_tensor(staging.layout)
    staging_smem.fill(0)
    cute.arch.sync_threads()

    producer_group = cutlass.pipeline.CooperativeGroup(
        cutlass.pipeline.Agent.Thread, 32
    )
    consumer_group = cutlass.pipeline.CooperativeGroup(
        cutlass.pipeline.Agent.Thread, 32
    )

    # 多级流水线：stages 个buffer，生产者最多可领先消费者 stages-1 步
    pipe = cutlass.pipeline.PipelineAsync.create(
        num_stages=stages,
        producer_group=producer_group,
        consumer_group=consumer_group,
        barrier_storage=storage.tma_mbar_ptr.data_ptr(),
    )

    producer, consumer = pipe.make_participants()

    # === 生产者（warp 0）===
    if warp_idx == 0:
        for i in cutlass.range(cute.size(res)):
            handle = producer.acquire_and_advance()
            # handle.index 在 0~stages-1 间轮转，写入对应的 SMEM buffer
            staging_smem[handle.index] = 1.0 * i
            handle.commit()
        producer.tail()

    # === 消费者（warp 1）===
    if warp_idx == 1:
        for i in cutlass.range(cute.size(res)):
            handle = consumer.wait_and_advance()
            res[i] = staging_smem[handle.index]
            handle.release()

    # 将 SMEM 中的最终状态拷贝回 GMEM，用于观察循环缓冲区的残留内容
    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        staging.store(staging_smem.load())


@cute.jit
def async_pipeline_staged(res: cute.Tensor, staging: cute.Tensor):
    stages = cute.size(staging)

    @cute.struct
    class SharedStorage:
        tma_mbar_ptr: cute.struct.MemRange[cutlass.Int64, stages * 2]
        staging_buffer: cute.struct.Align[
            cute.struct.MemRange[cutlass.Float32, stages], 1024
        ]

    async_pipeline_staged_kernel(SharedStorage, res, staging).launch(
        grid=(1, 1, 1), block=(64, 1, 1), smem=SharedStorage.size_in_bytes()
    )


res = torch.zeros((8,), device="cuda")
staging = torch.zeros((3,), device="cuda")
async_pipeline_staged(from_dlpack(res), from_dlpack(staging))
torch.cuda.synchronize()

In [30]:
res, staging

(tensor([0., 1., 2., 3., 4., 5., 6., 7.], device='cuda:0'),
 tensor([6., 7., 5.], device='cuda:0'))

### 补充：非阻塞 API

除了上述阻塞式的 `acquire` / `wait`，Pipeline 还提供非阻塞变体：

| 阻塞版本 | 非阻塞版本 | 行为差异 |
|---|---|---|
| `acquire()` | `try_acquire()` | 如果没有空buffer，立即返回失败而非阻塞 |
| `wait()` | `try_wait()` | 如果数据未就绪，立即返回失败而非阻塞 |

非阻塞 API 适用于生产者/消费者有独立计算可以执行的场景：先尝试获取/等待，如果未就绪则执行其他工作，稍后重试。

---

至此，异步流水线的核心概念（acquire -> commit -> wait -> release）和多级缓冲已经介绍完毕。接下来进入 GEMM 实现，你会看到 `PipelineTmaUmma` 和 `PipelineUmmaAsync` 这两种特化流水线，它们的 API 模式与本节完全一致。

## 二、GEMM 基础

### 理解 GEMM

GEMM 是线性代数和深度学习中最重要的操作之一。给定两个形状分别为 $(M, K)$ 和 $(N, K)$ 的二维 tensor A 和 B，GEMM 操作 $C = A \times B$ 定义为：

$
    C_{i,j} = \sum_{k=0}^{K-1} A_{i,k} * B_{j,k}
$

结果是一个形状为 $(M, N)$ 的二维 tensor C。

其中：
- $i \in [0, M)$ 表示 $A$ 和 $C$ 的行索引
- $j \in [0, N)$ 表示 $C$ 的列索引和 $B$ 的行索引
- $k \in [0, K)$ 表示 $A$ 和 $B$ 的列索引
- $A_{i,k}$、$B_{j,k}$ 和 $C_{i,j}$ 分别是 tensor $A$、$B$ 和 $C$ 在位置 $(i,k)$、$(j,k)$ 和 $(i,j)$ 处的元素

该操作有几个重要特性：

1. **可并行化**：每个元素可以独立计算。这有助于充分利用 GPU 中的 SM。
2. **数据可复用**：$C_{i,:}$（$C$ 的第 $i$ 行）需要相同的 $A_{i,:}$ 数据，而 $C_{:,j}$（$C$ 的第 $j$ 列）需要相同的 $B_{:,j}$ 数据。这种数据复用模式可以帮助减少 IO 压力。
3. **分块友好**：一组元素可以一起处理。每个块是整个 GEMM 的子问题。这有助于减少每个 SM 的 IO 压力。它使使用 MMA 指令加速计算成为可能。
4. **瓶颈灵活**：与 elementwise_add 不同，GEMM 的瓶颈因问题规模而异。让我们粗略计算 GEMM 的计算/内存比率：$ratio = \frac{M * N * K}{M*K + N*K + M*N} = \frac{1}{\frac{1}{N} + \frac{1}{M} + \frac{1}{K}}$。
它与 M、N 和 K 都相关。为了达到足够好的性能，我们需要针对不同的问题规模采用不同的策略。


### 朴素 GEMM

让我们从一个朴素实现开始，建立基准线，然后再探索优化方案。

![figure1](./images/blocked_gemm.svg "figure1: 分块 gemm")

首先，我们需要设置基本配置。

- io_dtype：tensor $A$、$B$ 和 $C$ 的数据类型。在大多数情况下，它也是 mma 指令的输入数据类型（有一些例外，如 TF32 数据类型、输入转换等）。

- acc_dtype：累加的数据类型。通常设置为 FP32 以避免溢出。由于 C 的数据类型可能与 acc_dtype 不同，acc 数据在存储之前需要转换为 io_dtype。

- mma_inst_shape_mnk：一条 tcgen05 mma 指令可以处理的形状。更多详情请参见 [PTX 文档 9.7.16.2.1. Matrix Shape](https://docs.nvidia.com/cuda/parallel-thread-execution/#tcgen05-matrix-shape)。
从一开始，我们选择最大的形状，因为它更容易达到 SOL。

- mma_tiler_mnk：GEMM kernel 通常实现为分块 GEMM（见图 1）。Mma tiler 是一个 CTA 或两个 CTA 将处理的块形状。是一个还是两个由 tcgen05 的发射粒度决定。更多详情请参见 [PTX 文档 9.7.16.5. Issue Granularity](https://docs.nvidia.com/cuda/parallel-thread-execution/#tcgen05-issue-granularity)。
从一开始，为简单起见，我们选择一个 CTA 来发射 tcgen05。

- threads_per_cta：一个 CTA 中需要使用的 thread 数量。为了充分利用 SM（流式多处理器），至少需要 128 个。

- ab_stages：该数字表示 TMA 在每个块计算之前可以加载多少个块。它通常受 smem 容量限制。对于 mma_tiler_mnk (128, 256, 64)，我们最多可以设置为 4。

- acc_stage：由于每个 CTA 只计算一个 acc 块并存储输出，该数字为 1。


In [31]:
io_dtype = cutlass.Float16
acc_dtype = cutlass.Float32
mma_inst_shape_mnk = (128, 256, 16)  # 单条 tcgen05 指令的形状
mma_tiler_mnk = (128, 256, 64)  # CTA tile 大小
threads_per_cta = 128

# 流水线阶段配置
ab_stages = 4
acc_stage = 1

接下来，让我们定义问题规模并初始化输入和输出 tensor，即 $A$、$B$ 和 $C$。

我们从一个典型的计算受限场景开始，即 8kx8kx8k。每个维度也足够大，可以避免分块量化问题。

In [32]:
m, n, k = 8192, 8192, 8192

# 创建 K 主序 tensor（torch tensor 是行主序的）
def make_tensors(mn, k, dtype):
    shape = (mn, k)
    return (
        torch.empty(*shape, dtype=torch.int32)
        .random_(-2, 2)
        .to(dtype=dtype, device="cuda")
    )

a = make_tensors(m, k, cutlass_torch.dtype(io_dtype))
b = make_tensors(n, k, cutlass_torch.dtype(io_dtype))
c = make_tensors(m, n, cutlass_torch.dtype(io_dtype))
a_tensor = (
    from_dlpack(a)
    .mark_layout_dynamic(leading_dim=1)
)
b_tensor = (
    from_dlpack(b)
    .mark_layout_dynamic(leading_dim=1)
)
c_tensor = (
    from_dlpack(c)
    .mark_layout_dynamic(leading_dim=1)
)

### Host 代码

GEMM kernel launch 前需要在 host 侧准备三样东西：**Tiled MMA**、**SMEM 布局**、**TMA 描述符**。

**Tiled MMA** 描述一条 MMA 指令的行为（数据类型、指令形状）以及指令在 tile 内如何重复。先用 `tcgen05.MmaF16BF16Op` 配置单条指令，再用 `cute.make_tiled_mma` 包装为 tiled 版本。

**SMEM 布局** 决定 A、B 在共享内存中的排列方式，需要匹配 MMA 指令的分区形状，并经过 swizzle 避免 bank conflict。`sm100_utils.make_smem_layout_a/b` 生成 4 维布局 (MMA_shape, M/N_repeat, K_repeat, Stages)，其中第 4 维是流水线 stage 数。TMA 只需要单 stage 的布局，所以用 `cute.select(mode=[0,1,2])` 去掉 stage 维度。

**TMA 描述符** 是硬件加速的异步内存搬运单元，一条指令即可将一个 tile 从 GMEM 搬到 SMEM。用 `CopyBulkTensorTileG2SOp` 定义搬运操作，再用 `make_tiled_tma_atom_A/B` 结合 GMEM tensor 和单 stage SMEM 布局，生成 tma_atom（搬运操作）和 tma_tensor（GMEM 视图）。

准备好这三样后，根据输出 tensor C 的形状计算 grid 大小（`ceil_div((M, N, 1), (128, 256))`，每个 CTA 负责一个 128x256 的输出块），然后 launch kernel。


In [ ]:
@cute.jit
def host_function(
    a: cute.Tensor,
    b: cute.Tensor,
    c: cute.Tensor,
    kernel: cutlass.Constexpr,
):
    # 使用 MmaF16BF16Op 配置一条 Blackwell tcgen05 MMA atomic 指令的全部属性
    op = tcgen05.MmaF16BF16Op(
        io_dtype,  # 输入数据类型
        acc_dtype,  # 累加数据类型
        mma_inst_shape_mnk,  # (128, 256, 16),单条 MMA 指令处理的形状。这是 tcgen05 支持的最大 MNK 形状，选最大是因为更容易打满硬件吞吐
        tcgen05.CtaGroup.ONE,  # 一个 CTA 独立发射 tcgen05 指令。Blackwell 支持两个 CTA 协作发射（CtaGroup.TWO），这里为了简单用单 CTA
        tcgen05.OperandSource.SMEM,  # 操作数源. A 和 B 的操作数来自 shared memory
        tcgen05.OperandMajorMode.K,  # A 和 B 都是 K 主序（即 K 维是连续的）, 这对应 torch 中的行主序 tensor
        tcgen05.OperandMajorMode.K,
    )
    # 构建 tiled MMA
    # tiled MMA 含义是在一个 tile（128x256x64）内，这条 128x256x16 的指令需要在 K 方向重复 64/16=4 次才能覆盖整个 tile 的 K 维。
    # 后续 kernel 中 `tiled_mma.get_slice(0)` 和 `partition_A/B/C` 都依赖这个 tiled MMA 来切分 tensor。
    # tiled_mma:
    #   ThrID: 1:0, Shape MNK: (128,256,16)
    #   TV Layout A: (1,(128,16)):(128,(1,128))
    #   TV Layout B: (1,(256,16)):(256,(1,256))
    #   TV Layout C: (1,(128,256)):(128,(1,128))
    tiled_mma = cute.make_tiled_mma(op)

    # 构建 A 和 B 的 SMEM 布局
    # make_smem_layout_a/b 是 CUTLASS 提供的 helper，它做了两件事：
    # - 根据 tiled MMA 的分区需求和 tile 形状，生成一个 4 维的 SMEM 布局：`(MMA_shape, M_repeat, K_repeat, Stages)`。
    #   前 3 维描述单个 stage 内的数据排列，第 4 维是流水线的 stage 数量（`ab_stages=4`）
    # - 内部自动做了 swizzle，来避免 bank conflict。
    a_smem_layout = sm100_utils.make_smem_layout_a(
        tiled_mma,
        mma_tiler_mnk,
        a.element_type,
        ab_stages,
    )
    b_smem_layout = sm100_utils.make_smem_layout_b(
        tiled_mma,
        mma_tiler_mnk,
        b.element_type,
        ab_stages,
    )
    # TMA 只需要单 stage 的布局，所以用 cute.select(mode=[0,1,2]) 去掉 stage 维度。
    # 返回的是一个 ComposedLayout，包含 outer（shape+stride）和 inner（swizzle 函数），kernel 中分配 SMEM tensor 时会用到。
    # a_smem_layout: S<3,4,3> o 0 o ((128,16),1,4,4):((64,1),0,16,8192)  -- 32768 FP16 = 64 KB
    # b_smem_layout: S<3,4,3> o 0 o ((256,16),1,4,4):((64,1),0,16,16384) -- 65536 FP16 = 128 KB
    # a+b smem total: 192 KB
    # a_smem_layout_one_stage: S<3,4,3> o 0 o ((128,16),1,4):((64,1),0,16) -- 去掉 stage 维
    # b_smem_layout_one_stage: S<3,4,3> o 0 o ((256,16),1,4):((64,1),0,16)
    # .outer = shape+stride: ((128,16),1,4,4):((64,1),0,16,8192)
    # .inner = swizzle: S<3,4,3>
    a_smem_layout_one_stage = cute.select(a_smem_layout, mode=[0, 1, 2])
    b_smem_layout_one_stage = cute.select(b_smem_layout, mode=[0, 1, 2])

    # 构建 TMA 描述符
    # CopyBulkTensorTileG2SOp 定义 TMA 的搬运方向：GMEM → SMEM（G2S），以及 CTA 粒度（ONE）。
    op = cute.nvgpu.cpasync.CopyBulkTensorTileG2SOp(tcgen05.CtaGroup.ONE)
    # `make_tiled_tma_atom_A` 把 TMA 操作、GMEM tensor、单 stage SMEM 布局、tile 形状和 tiled MMA 结合起来，其返回：
    # - `tma_atom_a`：TMA copy 原子操作，包含了硬件 TMA 描述符。kernel 中调用 cute.copy(tma_atom_a, ...) 时，底层就是发射一条 TMA 指令
    # - `a_tma_tensor`：GMEM tensor a/b 经过 TMA 视角重新 partition 后的视图。
    a_tma_atom, a_tma_tensor = cute.nvgpu.make_tiled_tma_atom_A(
        op,
        a,
        a_smem_layout_one_stage,
        mma_tiler_mnk,
        tiled_mma,
    )
    b_tma_atom, b_tma_tensor = cute.nvgpu.make_tiled_tma_atom_B(
        op,
        b,
        b_smem_layout_one_stage,
        mma_tiler_mnk,
        tiled_mma,
    )
    # a_tma_atom: ThrID=1:0, TV Src/Dst=(1,8192):(0,1), value_type=f16
    #   一条 TMA 指令搬运 8192 个 FP16 = 128x64 = 16KB（A 的一个 tile）
    # b_tma_atom: ThrID=1:0, TV Src/Dst=(1,16384):(0,1), value_type=f16
    #   一条 TMA 指令搬运 16384 个 FP16 = 256x64 = 32KB（B 的一个 tile）
    # a_tma_tensor layout: (?,?):(1@1,1@0) -- 动态 shape，TMA 视角的 GMEM 视图
    # a/b/c input tensor layout: (?,?):(?{i64},1) -- K 维 stride=1（K 主序），M/N 维 stride 动态

    # 启动 kernel
    # `c.layout.shape` 是 (M, N)，拼上 1 变成 (M, N, 1)。
    # `mma_tiler_mnk[:2]` 是 (128, 256)。`ceil_div` 算出 grid 形状为 `(ceil(M/128), ceil(N/256), 1)`。
    # 对于 8192x8192 的矩阵乘法，grid 是 (64, 32, 1)，一共 2048 个 CTA，每个负责输出矩阵的一个 128x256 块。
    # grid_shape: (ceil(M/128), ceil(N/256), 1) = (64, 32, 1) for 8192x8192, 共 2048 CTA
    grid_shape = cute.ceil_div((*c.layout.shape, 1), mma_tiler_mnk[:2])
    kernel(
        tiled_mma,
        a_tma_atom,
        a_tma_tensor,
        b_tma_atom,
        b_tma_tensor,
        c,
        a_smem_layout,
        b_smem_layout,
    ).launch(
        grid=grid_shape,
        block=(threads_per_cta, 1, 1), # 每个 CTA 用 128 个 thread（`threads_per_cta=128`，即 4 个 warp）。
    )

### Kernel 代码

Kernel 代码分为 Prologue、Mainloop、Epilogue 三部分。

**序言（Prologue）**：第一条 MMA 指令之前的阶段。它通常定义、获取、分配、分区或计算必要的组件（如下所列）。此外，在第一个 MMA 之前加载多个阶段的数据以帮助隐藏 GMEM 延迟。
   - 索引
     * `block_idx` (bidx, bidy)：grid 中的 block 索引
     * `mma_coord_mnk`：当前 MMA 单元将计算哪个块的位置（详见图 1）
     * `thread_idx` (tidx)：block 内的 thread 索引（0 到 threads_per_cta - 1）。我们需要这个来为 block 中的每个 thread 切片 tensor 内存的分区（详见 [PTX 文档 9.7.16.2.3.1 Memory Layout](https://docs.nvidia.com/cuda/parallel-thread-execution/#tcgen05-memory-layout)）
     * `warp_idx`：由于 TMA 和 tcgen05.mma 只需要一个 thread 发射，一些代码只需要由 warp 0 执行
   - 分配
     * `smem`（storage, sA, sB）：为流水线分配必要的 smem 使用量，A/B smem tensor 作为 tcgen05.mma 的输入
     * `tmem`：为 Acc 分配必要的 tmem 使用量
   - 流水线（更多详情请参见本文第一节）
     * `PipelineTmaUmma`：Tma 和 tcgen05.mma 单元是异步的。PipelineTmaUmma 帮助通知：1. 当 TMA 填满 A/B 缓冲区时通知 tcgen05.mma；2. 当 tcgen05.mma 消费完 A/B 缓冲区时通知 TMA
     * `PipelineUmmaAsync`：当 tcgen05.mma 完成累加且 Acc 准备好时帮助 thread
     * `Barrier 初始化`：barrier 初始化工作在流水线创建函数内部完成
   - 分区
     * `local_tile`：根据 `mma_tiler_mnk` 和 `mma_coord_mnk` 获取当前 CTA 负责的 A/B/C GMEM tensor 子块。`proj` 同时作用于 `tiler` 和 `coord`，`1` 的位置保留、`None` 的位置丢弃。注意 `proj` 中的 `None`（丢弃，该维度与当前 tensor 无关）和 `coord` 中的 `None`（不索引，保留该维度所有 tile）含义不同
     * `TMA`：从每条 TMA 指令获取 tensor 视图
     * `MMA`：从每条 tcgen05.mma 指令获取 tensor 视图
   - TMA 描述符预取
     * `cpasync.prefetch_descriptor`：帮助缩短访问 tma 描述符的延迟，即 tma_atom_a、tma_atom_b

**主循环（Mainloop）**：执行 GEMM 主要计算的阶段。它通常组织为一个循环，在 K 维度上迭代块进行累加。循环体包含：
    - `数据预取`，在当前 K 块之前以固定步幅（ab_stage - 1）预取
    - 当前 K 块的 `MMA 计算`

**尾声（Epilogue）**：MMA 指令完成累加后的阶段。它通常包含：
    - `分区`：从 epi tiler（acc 子分块）和每条 tcgen05.ld 指令获取 tensor 视图
    - `Acc 获取`：从 tensor memory 加载数据到寄存器
    - `融合和数据类型转换`：在 C 上融合一些操作（可选）；如果输出类型与 acc 类型不同则进行数据类型转换
    - `放弃 tmem 分配许可`：为后续启动的 kernel 提供许可
    - `存储`：TMA 或 st.global 存储输出
    - `TMEM 释放`：为 Acc 缓冲区释放 tmem
    
    通常，我们对 acc 缓冲区进行子分块以节省寄存器和 smem 资源（如果使用 TMA 存储 C）。对于我们的 mma_tiler (128, 256)，如果不进行子分块，每个 thread 需要 256 个寄存器。此外，交错发射 tcgen05.ld、数据转换和 st.global 可以获得更好的指令级并行性。

``` python
        for i in cutlass.range(cute.size(tDtC, mode=[2])):
            cute.copy(tmem_tiled_copy, tDtC[None, None, i], tCrAcc)
            tCrC.store(tCrAcc.load().to(io_dtype))
            cute.autovec_copy(tCrC, tDgC[None, None, i])
```

补充：**CuTe/CUTLASS Tensor 命名规则**

代码中的变量名遵循 CuTe/CUTLASS 的标准命名约定（参见 [0x_gemm_tutorial.md](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/cute/0x_gemm_tutorial.md)）。没有 `t` 前缀的是 partition 之前的 tensor，前缀字母表示内存空间：`m` = 完整矩阵（matrix），`g` = local_tile 后的 GMEM 子块，`s` = SMEM，`r` = Register，`t` = TMEM，后缀字母表示操作数（A、B、C）。例如 `mA` 是完整的 GMEM tensor A，`gA` 是 local_tile 后当前 CTA 负责的 GMEM 子块，`sA` 是 SMEM 中的 A buffer。

经过 partition 后的变量名格式为 **`tXyZ`**，其中 `tX` 是一个整体，表示分区模式（partitioning pattern），`y` 是内存空间，`Z` 是操作数。读作"分区模式 `tX` 应用到 tensor `yZ`"。`tX` 中的 `X` 标识使用了哪个 partitioner（分区器），不同的 partition 操作产生不同的 `X`：

- `tC` = MMA 分区模式（`thr_mma.partition_A/B/C` 统一由输出 C 的分区逻辑驱动，所以用 `C` 标识）
- `tA` = TMA-A 或 Copy-A 的分区模式
- `tB` = TMA-B 或 Copy-B 的分区模式
- `tD` = TMEM Copy 的分区模式

例如 `tCgA` = MMA 分区模式应用到 GMEM 的 A，`tAsA` = TMA-A 分区模式应用到 SMEM 的 A，`tDtC` = TMEM Copy 分区模式应用到 TMEM 的 C。对同一 tensor 使用相同的分区前缀（如 `tA` 同时应用于 `sA` 和 `gA`），可以保证 `cute.copy(tAsA, tAgA)` 中源和目标的逻辑一致性。


In [ ]:
@cute.struct
class SharedStorage:
    ab_mbar_ptr: cute.struct.MemRange[cutlass.Int64, ab_stages * 2]   # TMA-MMA 流水线 barrier
    acc_mbar_ptr: cute.struct.MemRange[cutlass.Int64, acc_stage * 2]  # 累加器流水线 barrier
    tmem_holding_buf: cutlass.Int32                                   # TMEM 分配辅助


@cute.kernel
def kernel(
    tiled_mma: cute.TiledMma,       # MMA 操作的配置（指令形状 + tile 内重复方式）
    tma_atom_a: cute.CopyAtom,      # A 的 TMA copy 原子操作
    mA_mkl: cute.Tensor,            # A 的 TMA tensor (GMEM)，形状 (M, K, L)，L 是 batch
    tma_atom_b: cute.CopyAtom,      # B 的 TMA copy 原子操作
    mB_nkl: cute.Tensor,            # B 的 TMA tensor (GMEM)，形状 (N, K, L)
    mC_mnl: cute.Tensor,            # C 的 GMEM tensor，形状 (M, N, L)
    a_smem_layout: cute.ComposedLayout,  # A 在 SMEM 中的布局（含 swizzle）
    b_smem_layout: cute.ComposedLayout,  # B 在 SMEM 中的布局（含 swizzle）
):
    #
    # 1. Prologue
    #

    # 当前 thread/warp/block 坐标
    tidx, _, _ = cute.arch.thread_idx()          # 当前 thread 在 block 内的 ID, 0~127, 用于后续按 thread 切分数据。
    warp_idx = cute.arch.warp_idx()              # 当前 warp 编号（tidx // 32）
    # make_warp_uniform 是因为编译器可能不确定同一 warp 内所有 thread 的 warp_idx 是否相同，加上这个编译器可以做分支优化
    warp_idx = cute.arch.make_warp_uniform(warp_idx)  # 保证 warp 内所有 thread 看到相同值
    bidx, bidy, _ = cute.arch.block_idx()        # 当前 block 在 grid 中的坐标
    mma_coord_mnk = (bidx, bidy, None)           # 本 CTA 负责的 (M_tile, N_tile, 全部K)，每个 CTA 负责完整的 K 维累加

    # 分配 SMEM
    smem = cutlass.utils.SmemAllocator()
    # SharedStorage 结构体定义了 barrier 和辅助变量的 SMEM 空间。
    storage = smem.allocate(SharedStorage)
    # sA 和 sB 是额外分配的 SMEM tensor，用于存放从 GMEM 搬来的 A 和 B 数据。
    # a_smem_layout 的类型是 cute.ComposedLayout，即复合布局。它由两部分组成：
    # - outer, 外层布局（shape + stride）,描述 tensor 的逻辑形状和在内存中的步长排列方式，即"数据怎么摆放"
    # - inner, 内层变换（swizzle 函数）,一个地址变换函数，对 outer 产生的线性地址做 bit-level 重排，用于消除 bank conflict
    sA = smem.allocate_tensor(
        element_type=io_dtype,
        layout=a_smem_layout.outer,   # 实际的 shape+stride
        byte_alignment=128,
        swizzle=a_smem_layout.inner,  # swizzle 函数，避免 bank conflict
    )
    sB = smem.allocate_tensor(
        element_type=io_dtype,
        layout=b_smem_layout.outer,
        byte_alignment=128,
        swizzle=b_smem_layout.inner,
    )
    # sA layout: ((128,16),1,4,4):((64,1),0,16,8192) -- 32768 FP16 = 64 KB
    # sB layout: ((256,16),1,4,4):((64,1),0,16,16384) -- 65536 FP16 = 128 KB
    # sA+sB = 192 KB (+ SharedStorage 88 bytes for barriers)

    # 分配 TMEM
    # NamedBarrier：创建一个命名屏障（barrier_id=1），参与线程数为 threads_per_cta（128）。
    # TMEM 分配只由 warp 0 中的一个线程执行，但分配完成后所有线程都需要拿到分配结果（TMEM 起始地址），所以需要这个屏障做 CTA 内同步。
    tmem_alloc_barrier = pipeline.NamedBarrier(
        barrier_id=1,
        num_threads=threads_per_cta,
    )
    # storage.tmem_holding_buf 是 SMEM 中一个 Int32 变量，
    # 用作"中转"——warp 0 执行分配后将 TMEM 起始地址写入这个 SMEM 位置，其他线程通过屏障同步后从这里读取。
    # barrier_for_retrieve 指定了上面的 NamedBarrier，确保所有线程在调用 retrieve_ptr() 前能看到正确的值。
    tmem = utils.TmemAllocator(
        storage.tmem_holding_buf,
        barrier_for_retrieve=tmem_alloc_barrier,
    )
    # 请求分配 512 列 TMEM。对于 mma_tiler_mnk = (128, 256, 64)，累加器形状为 128x256，
    # 以 FP32 存储。TMEM 每列有 128 行，每行存 16 bit。FP32 = 32 bit = 2 个 16b 单元，
    # 所以 256 列 FP32 数据需要 256*2 = 512 列 TMEM。
    num_tmem_cols = 512
    tmem.allocate(num_tmem_cols)

    # 预取 tma 描述符
    # prefetch_descriptor 做的是预取 TMA 描述符（descriptor），不是预取数据本身。
    # TMA 描述符是一个元数据结构，保存在常量内存（constant memory）或全局内存中，描述了源 tensor 的基地址、形状、stride、数据类型等信息。
    # 当后续执行 cute.copy(tma_atom_a, ...) 发起真正的 TMA 数据搬运时，硬件需要先读取这个描述符才能知道"从哪里搬、搬多少"。
    # 预取描述符的作用是提前将描述符加载到缓存中，减少后续第一次 TMA copy 的启动延迟。
    # 在这个 kernel 中，整个主循环的 TMA 加载和 MMA 计算都只由 warp 0 执行，所以预取描述符也只由 warp 0 执行。
    if warp_idx == 0:
        cpasync.prefetch_descriptor(tma_atom_a)
        cpasync.prefetch_descriptor(tma_atom_b)


    # 创建 PipelineTmaUmma 和 PipelineUmmaAsync 两条流水线，协调三个异步单元之间的同步
    # num_tma_copy_bytes 传给 tx_count，是每个 stage TMA 要搬运的总字节数（A 的一个 stage + B 的一个 stage）。
    # 这个值告诉硬件 barrier 等到这么多字节都到了 SMEM，才算一个 stage 的数据就绪。
    num_tma_copy_bytes = cute.size_in_bytes(
        io_dtype, cute.select(a_smem_layout, mode=[0, 1, 2])
    ) + cute.size_in_bytes(io_dtype, cute.select(b_smem_layout, mode=[0, 1, 2]))
    # num_tma_copy_bytes = A 一个 stage (16384 = 16KB) + B 一个 stage (32768 = 32KB) = 49152 = 48KB
    # PipelineTmaUmma（AB 流水线）协调 TMA（生产者）和 MMA（消费者）。
    # TMA 把 A/B 数据搬到 SMEM，MMA 从 SMEM 读数据做计算。4 个 stage 意味着 TMA 最多可以领先 MMA 3 步。
    ab_producer, ab_consumer = pipeline.PipelineTmaUmma.create(
        num_stages=ab_stages,
        producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        consumer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        tx_count=num_tma_copy_bytes,
        barrier_storage=storage.ab_mbar_ptr.data_ptr(),
    ).make_participants()
    # PipelineUmmaAsync（Acc 流水线）协调 MMA（生产者）和 epilogue thread（消费者）。
    # MMA 在 累加完所有 K 块后，通知 epilogue 可以读取累加器了。只有 1 个 stage（acc_stage=1），因为每个 CTA 只有一份累加器。
    # consumer_group 指定了 `threads_per_cta` 个 thread，因为 epilogue 阶段所有 128 个 thread 都参与数据搬出。
    acc_producer, acc_consumer = pipeline.PipelineUmmaAsync.create(
        num_stages=acc_stage,
        producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        consumer_group=pipeline.CooperativeGroup(
            pipeline.Agent.Thread, threads_per_cta
        ),
        barrier_storage=storage.acc_mbar_ptr.data_ptr(),
    ).make_participants()

    # tensor 分区：用 local_tile 从全局 tensor 中切出当前 CTA 负责的子块。
    #
    # - mma_tiler_mnk = (128, 256, 64) 是 CTA 级别的 tile 大小，定义了单个 CTA 在 M/N/K 三个维度上处理的数据量。
    # - mma_coord_mnk = (bidx, bidy, None) 是当前 CTA 在 tile 网格中的坐标，告诉 local_tile 该取哪一块。
    #   bidx 选 M 方向的第几个 tile，bidy 选 N 方向，None 表示 K 方向保留所有 K tile，留给 mainloop 逐个迭代。
    # - proj 同时作用于 tiler 和 coord，把 1 对应位置保留、None 对应位置丢弃。
    #   tiler 有 3 个维度 (M, N, K)，但 tensor A (M, K, L) 维度中没有 N，proj 的作用就是投影维度。
    # 
    # A (M,K,L): proj=(1,None,1) 保留 M+K, 丢弃 N -> tiler'=(128,64), coord'=(bidx, None)
    # coord 中 K 维度为 None 表示保留所有 K，因此加上第三维 num_k_tiles, gB 同理, gC 因为没有 K 维度, 所以最后只有两维。
    # (bM, bK, RestK), (128, 64, num_k_tiles=?)
    gA = cute.local_tile(mA_mkl, mma_tiler_mnk, mma_coord_mnk, proj=(1, None, 1))
    # B (N,K,L): proj=(None,1,1) 保留 N+K, 丢弃 M -> tiler'=(256,64), coord'=(bidy, None)
    # (bN, bK, RestK), (256, 64, num_k_tiles=?)
    gB = cute.local_tile(mB_nkl, mma_tiler_mnk, mma_coord_mnk, proj=(None, 1, 1))
    # C (M,N,L): proj=(1,1,None) 保留 M+N, 丢弃 K -> tiler'=(128,256), coord'=(bidx, bidy)
    # (bM, bN): (128, 256)
    gC = cute.local_tile(mC_mnl, mma_tiler_mnk, mma_coord_mnk, proj=(1, 1, None))

    # MMA 分区：按 MMA 指令的数据消费模式重新组织 tensor。
    # 变量命名规则 tXyZ：分区模式 X 应用到内存空间 y 中的操作数 Z。
    #
    # tcgen05 是 warp-level 指令, Blackwell 上操作数在 SMEM/TMEM 中，thread 间视图相同, 参数 0 表示取第 0 个 thread 的切片视图。
    thr_mma = tiled_mma.get_slice(0)
    # partition_A/B/C：将 gA/gB/gC 从物理布局 (M, K, ...) 重新组织成 MMA 指令视角的逻辑布局。
    # tcgen05.mma 的 partition 模式由输出 C 的分区决定，保证同一条指令消费的 A 行和 B 列恰好产出 C 中对应的元素块。
    # 所以 CuTe 统一用 C 的分区模式来 partition 所有操作数，前缀 tC 表示"用 C 的分区逻辑"。
    #
    # 变换过程分两步：
    #   1) 按单条 MMA 指令形状 (128, 256, 16) 分块 -> 产生第一个维度 MMA
    #   2) 计算 tile 内需要重复多少次 -> 产生 MMA_M/MMA_N/MMA_K 等"重复"维度
    #
    # 具体尺寸推算（以当前配置为例）：
    #   mma_inst = (128, 256, 16), tile = (128, 256, 64), K_total = 8192
    #
    #   tCgA: gA (128, 64, 128) -> ((128,16), MMA_M=128/128=1, MMA_K=64/16=4, RestK=8192/64=128)
    #   tCgB: gB (256, 64, 128) -> ((256,16), MMA_N=256/256=1, MMA_K=64/16=4, RestK=8192/64=128)
    #   tCgC: gC (128, 256)     -> ((128,256), MMA_M=1, MMA_N=1)
    #
    # tCgA/tCgB的用途是传给后面的 tma_partition, 告诉 TMA "GMEM 数据按 MMA 的视角看是什么样的"。
    tCgA = thr_mma.partition_A(gA)          # ((128,16), 1, 4, ?), stride: ((1@1,1@0), 0, 16@0, 64@0)
    tCgB = thr_mma.partition_B(gB)          # ((256,16), 1, 4, ?), stride: ((1@1,1@0), 0, 16@0, 64@0)
    # tCgC 用于 epilogue 阶段的 GMEM 存储分区
    tCgC = thr_mma.partition_C(gC)          # ((128,256), 1, 1), stride: ((?{i64},1), 0, 0)
    # make_fragment_A/B：为 A/B 创建 MMA 视角的 "fragment" tensor。
    # 命名中 r 沿用 CuTe 惯例表示 register，但 tcgen05 的 A/B 操作数直接从 SMEM 读取, 所以这里实际是对 sA/sB 重新 partition.
    #
    # sA 的 layout 是 (MMA_shape, M_repeat, K_repeat, stages=4)，make_fragment_A 对前 3 维做 MMA partition，第 4 维 stages 直接保留。
    # mainloop 中通过 tCrA[(None, None, k_block_idx, stage_idx)] 同时索引 K 子块和流水线 stage。
    tCrA = tiled_mma.make_fragment_A(sA)    # (1, 1, 4, 4):(0, 0, 2, 1024) -- tcgen05 的 A 操作数直接从 SMEM 读，不做 thread 级分区，前两维为 1
    tCrB = tiled_mma.make_fragment_B(sB)    # (1, 1, 4, 4):(0, 0, 2, 2048) -- 同上，后两维为 (MMA_K=4, ab_stages=4)

    # partition_shape_C 根据 tile 的 (M, N) = (128, 256) 计算出 MMA 分区后的形状。
    acc_shape = tiled_mma.partition_shape_C(mma_tiler_mnk[:2])  # ((128,256), MMA_M=1, MMA_N=1)
    # make_fragment_C：创建累加器 tensor，存放在 TMEM 中。命名中的 t 代表 TMEM。
    # 此处创建的 fragment 初始指向占位地址，后面用 cute.make_tensor(tmem_ptr, ...) 替换为真正的 TMEM 指针。
    # acc_shape: ((128,256), 1, 1)
    # tCtAcc layout: ((128,256), 1, 1):((65536,1), 0, 0) -- 32768 FP32 = 128 KB
    tCtAcc = tiled_mma.make_fragment_C(acc_shape)

    # TMA 分区：把 GMEM/SMEM tensor 按 TMA 搬运单元的视角重新分区。
    #
    # group_modes 做的是降维
    # group_modes(sA, 0, 3)：sA 是 (MMA_shape, M_repeat, K_repeat, stages)，合并前 3 维 → (tile_data, stages=4)。
    # group_modes(tCgA, 0, 3)：tCgA 是 ((128,16), MMA_M=1, MMA_K=4, num_k_tiles)，合并前 3 维 → (tile_data, num_k_tiles=128)。
    #
    # 为什么传 tCgA 而不是 gA？TMA 搬运进 SMEM 的数据排列方式，必须和后续 MMA 读取的布局一致。
    #
    # tAsA:((8192,1), ab_stages=4)  # 8192 = 128*64 个 FP16 元素 = 16KB，即一个完整 A tile；4 个 stages
    # tAgA:(((64,128),1), num_k_tiles) # 重排为 (K_tile=64, M_tile=128)，第二维索引 GMEM 中的第几个 K-tile
    tAsA, tAgA = cute.nvgpu.cpasync.tma_partition(
        tma_atom_a,
        0,
        cute.make_layout(1),
        cute.group_modes(sA, 0, 3),
        cute.group_modes(tCgA, 0, 3),
    )
    # tBsB:((16384,1), ab_stages=4)  # 16384 = 256*64 个 FP16 = 32KB
    # tBgB:(((64,256),1), num_k_tiles)
    tBsB, tBgB = cute.nvgpu.cpasync.tma_partition(
        tma_atom_b,
        0,
        cute.make_layout(1),
        cute.group_modes(sB, 0, 3),
        cute.group_modes(tCgB, 0, 3),
    )
    # group_modes(sA, 0, 3): (((128,16),1,4), 4):(((64,1),0,16), 8192) -- 合并前 3 维为 tile_data, 保留 stages=4
    # group_modes(tCgA, 0, 3): (((128,16),1,4), ?):(((1@1,1@0),0,16@0), 64@0)
    # tAsA: ((8192,1), 4):((1,0), 8192) -- 每 stage 8192 FP16 = 16 KB
    # tAgA: (((64,128),1), ?):(((1@0,1@1),0), 64@0)
    # tBsB: ((16384,1), 4):((1,0), 16384) -- 每 stage 16384 FP16 = 32 KB
    # tBgB: (((64,256),1), ?):(((1@0,1@1),0), 64@0)

    # 总结下三种 partition 的关系，其实是同一份 A 数据在不同阶段的视角：
    #   tCgA  (MMA 视角看 GMEM)：按 MMA 指令形状切分 gA，产出 ((128,16), 1, 4, num_k_tiles), 用于 tma partition
    #   tCrA  (MMA 视角看 SMEM)：make_fragment_A 对 sA 做 MMA partition，产出 (1, 1, 4, ab_stages=4)
    #         tcgen05 的 A/B 操作数直接从 SMEM 整块读取，不做 thread 级分区，所以前两维折叠为 1。
    #         后两维 (MMA_K=4, stages=4) 用于在 mainloop 中索引: tCrA[(None, None, k_block_idx, stage_idx)]
    #   tAgA  (TMA 视角看 GMEM)：TMA 搬运的源地址，(((64,128),1), num_k_tiles)
    #   tAsA  (TMA 视角看 SMEM)：TMA 搬运的目标地址，((8192,1), ab_stages=4)
    # 数据流：TMA 从 GMEM 搬到 SMEM (tAgA -> tAsA)，MMA 从 SMEM 读取计算 (tCrA)。

    # 等待 TMEM 分配完成（warp 0 分配，其他 warp 在此等待）。
    tmem.wait_for_alloc()
    # 取出 TMEM 的起始地址，按 acc_dtype的 FP32 类型解释。
    tmem_ptr = tmem.retrieve_ptr(acc_dtype)
    # 把占位的累加器 fragment 替换成真正指向 TMEM 的 tensor，layout 不变，只换了底层指针。
    tCtAcc = cute.make_tensor(tmem_ptr, tCtAcc.layout)

    # 累加器 tCtAcc (128x256 FP32 = 128KB) 在 mainloop 阶段存放在 TMEM 中，不占寄存器资源，
    # 但 epilogue 需要从 TMEM 搬到寄存器（tcgen05.ld）再转换精度写入 GMEM，如果一次性搬运整个 tCtAcc，分摊到 128 threads，
    # 每 thread 需要 256 个 FP32 寄存器来暂存，超出了 CUDA 每线程 255 个寄存器的编译器上限（maxrregcount）。
    # 因此对 tCtAcc 进行子分块，把维度 N 分成 subtile_cnt=4 份，即 EpiTile = 32KB，每个 thread 占用 64 个寄存器
    subtile_cnt = 4
    # epi_tiler = ((128, 64),)
    epi_tiler = (
        (cute.size(tCtAcc, mode=[0, 0]), cute.size(tCtAcc, mode=[0, 1]) // subtile_cnt),
    )
    # zipped_divide: 按 epi_tiler 分块，生成 (EpiTile=(128,64), NumTiles=4) 二维视图。
    # tCtAcc_epi 和 gC_epi 用相同的 epi_tiler 分块，保证子块一一对应：
    #   tCtAcc_epi[..., i] 是 TMEM 中第 i 个子块（数据来源）
    #   gC_epi[..., i]     是 GMEM 中第 i 个子块（写入目标）
    # epilogue 时, 循环 for i in range(4) : TMEM[i] -> Register -> FP16 -> GMEM[i]
    tCtAcc_epi = cute.zipped_divide(tCtAcc, epi_tiler)
    gC_epi = cute.zipped_divide(tCgC, epi_tiler)
    # epi_tiler = ((128, 64),) -- 每个子块 128x64 FP32 = 32 KB, 分摊到 128 threads = 64 regs/thread
    # tCtAcc_epi: (((128,64)), ((1,4),1,1)):(((65536,1)), ((0,64),0,0))
    # gC_epi: (((128,64)), ((1,4),1,1)):(((?{i64},1)), ((0,64),0,0))

    # TMEM Copy 配置：定义如何从 TMEM 搬运数据到寄存器。
    # tcgen05.Ld32x32bOp：对应 PTX 的 tcgen05.ld.sync.aligned.32x32b，加载 32 列 TMEM，每列 32 bit。
    # Repetition.x64：对应 PTX 的 .x64，每个 thread 获得 64 个 32-bit 寄存器。
    # 128 threads x 64 regs x 4 bytes = 32KB，恰好覆盖一个 epilogue 子块 (128x64 FP32 = 32KB)。
    tmem_atom = cute.make_copy_atom(
        tcgen05.Ld32x32bOp(tcgen05.Repetition.x64),
        cutlass.Float32,
    )
    # make_tmem_copy：把 atom 铺展到一个 epi_tile (128x64 FP32) 上，生成覆盖整个子块的 tiled copy。
    # tCtAcc_epi[None, 0] 是第 0 个子块，用它的 layout 来确定如何铺展。
    tmem_tiled_copy = tcgen05.make_tmem_copy(tmem_atom, tCtAcc_epi[None, 0])
    # get_slice(tidx)：按当前 thread 的 ID 切分出该 thread 负责搬运的数据分区。
    # 与 MMA 的 get_slice(0) 不同，TMEM copy 是 thread-level 操作，每个 thread 搬运不同的数据。
    tmem_thr_copy = tmem_tiled_copy.get_slice(tidx)

    # tDtC = "TMEM copy 分区(D) 应用到 TMEM(t) 中的 C", 即 TMEM copy 视角看 TMEM 中的累加器。
    # tDgC = "TMEM copy 分区(D) 应用到 GMEM(g) 中的 C", 即 TMEM copy 视角看 GMEM。
    # 两者均为 (TmemCpy, NumTmemCpy, NumTiles=4)
    # TmemCpy是 单条 TMEM load 指令搬运的数据量（每 thread), NumTmemCpy 是需要多少条指令才能覆盖一个子块
    # tmem_tiled_copy: Tiler MN=((256,32):(32,1)), ThrID=32:1
    #   Copy Atom: TV Src=(32,2048):(0,1), TV Dst=(32,64):(64,1), f32
    # tDtC: (((64,32),1), 1, ((1,4),1,1)):(((1,65536),0), 0, ((0,64),0,0)) -- 每 thread 64 FP32 from TMEM
    # tDgC: ((64,1), 1, ((1,4),1,1)):((1,0), 0, ((0,64),0,0)) -- 每 thread 64 元素写入 GMEM
    # epilogue loop count = 4 (subtiles)
    tDtC = tmem_thr_copy.partition_S(tCtAcc_epi)
    tDgC = tmem_thr_copy.partition_D(gC_epi)

    # 寄存器缓冲区，一个子块的数据先从 TMEM 加载到 tCrAcc (FP32)，然后转换精度存入 tCrC (FP16)，最后从 tCrC 写入 GMEM。
    # shape 取 tDgC 的前两维 (TmemCpy, NumTmemCpy)，即单个子块中当前 thread 负责的数据量。
    # tCrAcc: ((64,1), 1):((1,0), 0) -- FP32 寄存器 buffer, 每 thread 64 个
    tCrAcc = cute.make_rmem_tensor(tDgC[None, None, 0].shape, acc_dtype)
    # tCrC: ((64,1), 1):((1,0), 0) -- FP16 寄存器 buffer, 每 thread 64 个
    tCrC = cute.make_rmem_tensor(tDgC[None, None, 0].shape, io_dtype)

    #
    # 2. 主循环
    #
    # num_k_tiles = K/tile_K = 8192/64 = 128 (动态值，编译时为 ?)
    # num_k_blocks = tile_K/inst_K = 64/16 = 4 (静态值)
    # total MMA instructions per CTA = 128 * 4 = 512
    num_k_tiles = cute.size(gA, mode=[2])
    if warp_idx == 0:
        # 等待空的累加器缓冲区
        acc_empty = acc_producer.acquire_and_advance()
        for k_tile_idx in cutlass.range(num_k_tiles):
            # 发射 TMA 加载
            ab_empty = ab_producer.acquire_and_advance()
            cute.copy(
                tma_atom_a,
                tAgA[(None, ab_empty.count)],
                tAsA[(None, ab_empty.index)],
                tma_bar_ptr=ab_empty.barrier,
            )
            cute.copy(
                tma_atom_b,
                tBgB[(None, ab_empty.count)],
                tBsB[(None, ab_empty.index)],
                tma_bar_ptr=ab_empty.barrier,
            )

            # 执行一个 K 块的 MMA 指令
            ab_full = ab_consumer.wait_and_advance()
            num_k_blocks = cute.size(tCrA, mode=[2])
            for k_block_idx in cutlass.range_constexpr(num_k_blocks):
                k_block_coord = (None, None, k_block_idx, ab_full.index)
                cute.gemm(
                    tiled_mma,
                    tCtAcc,
                    tCrA[k_block_coord],
                    tCrB[k_block_coord],
                    tCtAcc,
                )
                tiled_mma.set(tcgen05.Field.ACCUMULATE, True)

            # 通知 A/B 缓冲区已被消费, 准备好进行下一次加载
            ab_full.release()

        # 通知累加器已完全计算完成
        acc_empty.commit()

    #
    # 3. Epilogue
    #

    # 释放 TMEM 分配锁
    tmem.relinquish_alloc_permit()

    # 等待累加器缓冲区填满
    acc_full = acc_consumer.wait_and_advance()

    # TMEM -> RMEM -> GEMM
    # 子分块以获得更好的指令级并行性
    for i in cutlass.range(cute.size(tDtC, mode=[2])):
        cute.copy(tmem_tiled_copy, tDtC[None, None, i], tCrAcc)
        tCrC.store(tCrAcc.load().to(io_dtype))
        cute.autovec_copy(tCrC, tDgC[None, None, i])
    acc_full.release()

    # 释放 TMEM
    pipeline.sync(barrier_id=1)
    tmem.free(tmem_ptr)

## 三、性能分析和基准测试

为了理解和改进 kernel 的性能，我们需要测量其执行时间和计算吞吐量。以下是我们测量这些指标的基准测试工具：


In [35]:
def benchmark(callable, a_tensor, b_tensor, c_tensor):
    avg_time_us = cute.testing.benchmark(
        callable,
        kernel_arguments=cute.testing.JitArguments(a_tensor, b_tensor, c_tensor),
        warmup_iterations=1,
        iterations=1,
    )

    # 计算总浮点运算数：
    # - M * N * K * 2（FMA）
    total_float_ops = m * n * k * 2

    # 计算实际 TFlops
    achieved_tflops = total_float_ops / (avg_time_us * 1000000)  # TFlops

    # 打印结果
    # ------------
    print(f"性能指标:")
    print(f"-------------------")
    print(f"Kernel 执行时间: {avg_time_us:.4f} us")
    print(f"计算吞吐量: {achieved_tflops:.2f} tflops")

### 测试朴素 GEMM

你可以运行以下代码获取 Tflops，并使用 `torch.einsum` 作为参考来验证函数是否正确。

你应该能够达到大约 **450** TFlops。


In [39]:
# 为特定输入类型编译 kernel
naive_kernel = cute.compile(host_function, a_tensor, b_tensor, c_tensor, kernel)

# 运行 kernel
benchmark(naive_kernel, a_tensor, b_tensor, c_tensor)

# 验证结果
# -------------
# 将我们的 kernel 输出与 PyTorch 的原生实现进行比较
# 计算参考结果并验证
ref = (torch.einsum("mk,nk->mn", a.to(torch.float32), b.to(torch.float32))).cpu()
torch.testing.assert_close(
    c.cpu(), ref.to(cutlass_torch.dtype(io_dtype)), atol=1e-1, rtol=1e-05
)
print("验证通过!")

[Host] tiled_mma: Tiled MMA
  Thr Layout VMNK: (1,1,1,1):(0,0,0,0)
  Permutation MNK: (_,_,_)
MMA Atom
  ThrID:           1:0
  Shape MNK:       (128,256,16)
  TV Layout A:     (1,(128,16)):(128,(1,128))
  TV Layout B:     (1,(256,16)):(256,(1,256))
  TV Layout C:     (1,(128,256)):(128,(1,128))
[Host] a_smem_layout (full, 4D): S<3,4,3> o 0 o ((128,16),1,4,4):((64,1),0,16,8192)
[Host] b_smem_layout (full, 4D): S<3,4,3> o 0 o ((256,16),1,4,4):((64,1),0,16,16384)
[Host] a_smem_layout_one_stage: S<3,4,3> o 0 o ((128,16),1,4):((64,1),0,16)
[Host] b_smem_layout_one_stage: S<3,4,3> o 0 o ((256,16),1,4):((64,1),0,16)
[Host] a_smem_layout.outer: ((128,16),1,4,4):((64,1),0,16,8192)
[Host] a_smem_layout.inner: S<3,4,3>
[Host] a_smem_layout size (all stages): 32768 elements = 64 KB
[Host] b_smem_layout size (all stages): 65536 elements = 128 KB
[Host] a+b smem total: 192 KB
[Host] a_tma_atom: Copy Atom
  ThrID:         1:0
  TV Layout Src: (1,8192):(0,1)
  TV Layout Dst: (1,8192):(0,1)
  Value ty


软件流水线是一种在计算当前分块时预取下一个分块数据的技术，从而隐藏内存延迟。（详见本文第二节）。

## 四、启用软件流水线

正如我们之前所说，通常我们预取多个阶段（ab_stages - 2）的 A/B tensor 来隐藏 GMEM 延迟（见图 2）。深色区域表示 TMA/tcgen05.mma 的发射，浅色区域相应地表示延迟。
它可以使用 (ab\_stages - 1) * 单阶段 mma 时间来隐藏 GMEM 延迟。

![figure2](./images/software_pipelining_ab_stages_minus_2.svg "figure2: 预取 ab_stages_minus_2 的软件流水线")



要启用此策略，我们：
1. 在主循环之前编写一个循环进行预取
2. 在主循环内以固定步幅提前复制


``` python
## 预取 ab_stages - 2 块的 A/B
for stage in cutlass.range(ab_stages - 2):
    ab_empty = ab_producer.acquire_and_advance()
    cute.copy(...)

for k_tile_idx in cutlass.range(num_k_tiles):
    # 发射 TMA 加载
    if k_tile_idx + ab_stages - 2 < num_k_tiles:
        ab_empty = ab_producer.acquire_and_advance()
        cute.copy(...)
    # 执行一个 K 块的 MMA 指令
    ab_full = ab_consumer.wait_and_advance()
    cute.gemm(...)
    # 通知 A/B 缓冲区已被消费，准备好进行下一次加载
    ab_full.release()
```

对于 CuTeDSL，我们为 cutlass.range 提供了一个 `prefetch_stages` 属性。它帮助我们像通用模式一样编写代码，但像上面那样预取数据。

``` python
for k_tile_idx in cutlass.range(num_k_tiles, prefetch_stages=ab_stages - 2):
    # 发射 TMA 加载
    ab_empty = ab_producer.acquire_and_advance()
    cute.copy(...)
    # 执行一个 K 块的 MMA 指令
    ab_full = ab_consumer.wait_and_advance()
    cute.gemm(...)
    # 通知 A/B 缓冲区已被消费，准备好进行下一次加载
    ab_full.release()
```

图 3 解释了为什么我们预取 ab_stages - 2 而不是 ab_stages - 1。对于 ab_stages - 1，主循环内的每次 TMA 复制将在前一个 MMA 完成后发射。这将延迟下一个 MMA 的发射并导致两个块之间的气泡。

![figure3](./images/software_pipelining_ab_stages_minus_1.svg "figure3: 预取 ab_stages_minus_1 的软件流水线")

让我们测试启用预取后的性能。你可以达到大约 **880** TFlops。


In [ ]:
@cute.kernel
def kernel_with_prefetch(
    tiled_mma: cute.TiledMma,
    tma_atom_a: cute.CopyAtom,
    mA_mkl: cute.Tensor,
    tma_atom_b: cute.CopyAtom,
    mB_nkl: cute.Tensor,
    mC_mnl: cute.Tensor,
    a_smem_layout: cute.ComposedLayout,
    b_smem_layout: cute.ComposedLayout,
):
    #
    # 1. 准备参数
    #

    # 当前 thread/warp/block 坐标
    tidx, _, _ = cute.arch.thread_idx()
    warp_idx = cute.arch.warp_idx()
    warp_idx = cute.arch.make_warp_uniform(warp_idx)
    bidx, bidy, _ = cute.arch.block_idx()
    mma_coord_mnk = (bidx, bidy, None)

    # 分配 SMEM
    smem = cutlass.utils.SmemAllocator()
    storage = smem.allocate(SharedStorage)
    sA = smem.allocate_tensor(
        element_type=io_dtype,
        layout=a_smem_layout.outer,
        byte_alignment=128,
        swizzle=a_smem_layout.inner,
    )
    sB = smem.allocate_tensor(
        element_type=io_dtype,
        layout=b_smem_layout.outer,
        byte_alignment=128,
        swizzle=b_smem_layout.inner,
    )

    # 分配所有 TMEM 列
    tmem_alloc_barrier = pipeline.NamedBarrier(
        barrier_id=1,
        num_threads=threads_per_cta,
    )
    tmem = utils.TmemAllocator(
        storage.tmem_holding_buf,
        barrier_for_retrieve=tmem_alloc_barrier,
    )
    num_tmem_cols = 512
    tmem.allocate(num_tmem_cols)

    # 预取 tma 描述符
    if warp_idx == 0:
        cpasync.prefetch_descriptor(tma_atom_a)
        cpasync.prefetch_descriptor(tma_atom_b)

    # 流水线配置
    num_tma_copy_bytes = cute.size_in_bytes(
        io_dtype, cute.select(a_smem_layout, mode=[0, 1, 2])
    ) + cute.size_in_bytes(io_dtype, cute.select(b_smem_layout, mode=[0, 1, 2]))
    ab_producer, ab_consumer = pipeline.PipelineTmaUmma.create(
        num_stages=ab_stages,
        producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        consumer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        tx_count=num_tma_copy_bytes,
        barrier_storage=storage.ab_mbar_ptr.data_ptr(),
    ).make_participants()
    acc_producer, acc_consumer = pipeline.PipelineUmmaAsync.create(
        num_stages=acc_stage,
        producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
        consumer_group=pipeline.CooperativeGroup(
            pipeline.Agent.Thread, threads_per_cta
        ),
        barrier_storage=storage.acc_mbar_ptr.data_ptr(),
    ).make_participants()

    # 为 MMA 分区 tensor 并创建片段
    # (bM, bK, RestK)
    gA = cute.local_tile(mA_mkl, mma_tiler_mnk, mma_coord_mnk, proj=(1, None, 1))
    # (bN, bK, RestK)
    gB = cute.local_tile(mB_nkl, mma_tiler_mnk, mma_coord_mnk, proj=(None, 1, 1))
    # (bM, bN)
    gC = cute.local_tile(mC_mnl, mma_tiler_mnk, mma_coord_mnk, proj=(1, 1, None))
    thr_mma = tiled_mma.get_slice(0)
    # (MMA, MMA_M, MMA_K)
    tCgA = thr_mma.partition_A(gA)
    # (MMA, MMA_N, MMA_K)
    tCgB = thr_mma.partition_B(gB)
    # (MMA, MMA_M, MMA_N)
    tCgC = thr_mma.partition_C(gC)
    # (MMA, MMA_M, MMA_K)
    tCrA = tiled_mma.make_fragment_A(sA)
    # (MMA, MMA_N, MMA_K)
    tCrB = tiled_mma.make_fragment_B(sB)
    # (MMA, MMA_M, MMA_N)
    acc_shape = tiled_mma.partition_shape_C(mma_tiler_mnk[:2])
    # (MMA, MMA_M, MMA_N)
    tCtAcc = tiled_mma.make_fragment_C(acc_shape)
    # 为 TMA 分区 tensor；这需要为 MMA 分区的 tensor
    tAsA, tAgA = cute.nvgpu.cpasync.tma_partition(
        tma_atom_a,
        0,
        cute.make_layout(1),
        cute.group_modes(sA, 0, 3),
        cute.group_modes(tCgA, 0, 3),
    )
    tBsB, tBgB = cute.nvgpu.cpasync.tma_partition(
        tma_atom_b,
        0,
        cute.make_layout(1),
        cute.group_modes(sB, 0, 3),
        cute.group_modes(tCgB, 0, 3),
    )

    # 在检索分配的 TMEM 起始指针之前进行 CTA 范围同步
    # 只有 warp 0 进行分配，所以我们需要在检索 TMEM 起始地址之前同步
    tmem.wait_for_alloc()
    tmem_ptr = tmem.retrieve_ptr(acc_dtype)
    # 交换 tCtAcc 中的指针
    tCtAcc = cute.make_tensor(tmem_ptr, tCtAcc.layout)

    subtile_cnt = 4
    # (EpiTile)
    epi_tiler = (
        (cute.size(tCtAcc, mode=[0, 0]), cute.size(tCtAcc, mode=[0, 1]) // subtile_cnt),
    )
    # (EpiTile, NumTiles)
    tCtAcc_epi = cute.zipped_divide(tCtAcc, epi_tiler)
    # (EpiTile, NumTiles)
    gC_epi = cute.zipped_divide(tCgC, epi_tiler)

    # 每个 thread 加载 32x128 位
    tmem_atom = cute.make_copy_atom(
        tcgen05.Ld32x32bOp(tcgen05.Repetition.x64),
        cutlass.Float32,
    )
    tmem_tiled_copy = tcgen05.make_tmem_copy(tmem_atom, tCtAcc_epi[None, 0])
    tmem_thr_copy = tmem_tiled_copy.get_slice(tidx)

    # (TmemCpy,NumTmemCpy,NumTiles)
    tDtC = tmem_thr_copy.partition_S(tCtAcc_epi)
    # (TmemCpy,NumTmemCpy,NumTiles)
    tDgC = tmem_thr_copy.partition_D(gC_epi)

    # (TmemCpy,NumTmemCpy)
    tCrAcc = cute.make_rmem_tensor(tDgC[None, None, 0].shape, acc_dtype)
    # (TmemCpy,NumTmemCpy)
    tCrC = cute.make_rmem_tensor(tDgC[None, None, 0].shape, io_dtype)

    #
    # 2. 主循环
    #
    num_k_tiles = cute.size(gA, mode=[2])
    if warp_idx == 0:
        # 等待空的累加器缓冲区
        acc_empty = acc_producer.acquire_and_advance()
        # ======= 仅此行代码不同 ======= #
        for k_tile_idx in cutlass.range(num_k_tiles, prefetch_stages=ab_stages - 2):
            # 发射 TMA 加载
            ab_empty = ab_producer.acquire_and_advance()
            cute.copy(
                tma_atom_a,
                tAgA[(None, ab_empty.count)],
                tAsA[(None, ab_empty.index)],
                tma_bar_ptr=ab_empty.barrier,
            )
            cute.copy(
                tma_atom_b,
                tBgB[(None, ab_empty.count)],
                tBsB[(None, ab_empty.index)],
                tma_bar_ptr=ab_empty.barrier,
            )

            # 执行一个 K 块的 MMA 指令
            ab_full = ab_consumer.wait_and_advance()
            num_k_blocks = cute.size(tCrA, mode=[2])
            for k_block_idx in cutlass.range_constexpr(num_k_blocks):
                k_block_coord = (None, None, k_block_idx, ab_full.index)
                cute.gemm(
                    tiled_mma,
                    tCtAcc,
                    tCrA[k_block_coord],
                    tCrB[k_block_coord],
                    tCtAcc,
                )
                tiled_mma.set(tcgen05.Field.ACCUMULATE, True)

            # 通知 A/B 缓冲区已被消费，准备好进行下一次加载
            ab_full.release()

        # 通知累加器已完全计算完成
        acc_empty.commit()

    #
    # 3. 尾声
    #

    # 释放 TMEM 分配锁
    tmem.relinquish_alloc_permit()

    # 等待累加器缓冲区填满
    acc_full = acc_consumer.wait_and_advance()

    # TMEM -> RMEM -> GEMM
    # 子分块以获得更好的指令级并行性
    for i in cutlass.range(cute.size(tDtC, mode=[2])):
        cute.copy(tmem_tiled_copy, tDtC[None, None, i], tCrAcc)
        tCrC.store(tCrAcc.load().to(io_dtype))
        cute.autovec_copy(tCrC, tDgC[None, None, i])
    acc_full.release()

    # 释放 TMEM
    pipeline.sync(barrier_id=1)
    tmem.free(tmem_ptr)

In [15]:
# 为特定输入类型编译 kernel
prefetch_kernel = cute.compile(host_function, a_tensor, b_tensor, c_tensor, kernel_with_prefetch)

# 运行 kernel
benchmark(prefetch_kernel, a_tensor, b_tensor, c_tensor)

# 验证结果
# -------------
# 将我们的 kernel 输出与 PyTorch 的原生实现进行比较
# 计算参考结果并验证
ref = (torch.einsum("mk,nk->mn", a.to(torch.float32), b.to(torch.float32))).cpu()
torch.testing.assert_close(
    c.cpu(), ref.to(cutlass_torch.dtype(io_dtype)), atol=1e-1, rtol=1e-05
)
print("验证通过!")

性能指标:
-------------------
Kernel 执行时间: 1252.8160 us
计算吞吐量: 877.63 tflops
验证通过!


## 五、向量化存储输出

如果我们使用 NCU 来分析这个 kernel，每次 wave 切换时 TensorCore 利用率急剧下降。这是因为大量的 st.global.b16 存储指令排队占用了存储管线，反过来阻塞了后续 wave 的 MMA 发射。

CuTeDSL 需要对齐和可整除性来为 cute.copy 选择向量化指令，即从 st.global.b16	到 st.global.b128, 我们需要在 host 侧定义 cute tensor 时正确设置这些属性。


In [ ]:
# 与 Cell 16 的 a_tensor/b_tensor/c_tensor 相比，这里额外设置了两个属性：
#
# - assumed_align=32 告诉编译器这个 tensor 的基地址是 32 字节对齐的。
# 没有这个信息，编译器只能假设最保守的 2 字节对齐（FP16 元素大小），每次只能发射 st.global.b16 存一个元素。
# 有了 32 字节对齐保证，编译器可以生成 st.global.b128（16 字节）甚至更宽的向量化存储指令，一条指令写 8 个 FP16。
#
# - mark_compact_shape_dynamic(mode=1, divisibility=k) 告诉编译器 tensor 第 1 维（K 方向）的大小能被 k（8192）整除。
# 这对 epilogue 中 的向量化至关重要：编译器需要确认连续存储的元素数量能被向量宽度整除，否则无法安全地使用宽存储指令。
# 对 C tensor 来说，mode=1 是 N 方向，divisibility=n 同理。
#
# 即 epilogue 阶段 cute.autovec_copy 把寄存器中的数据写回 GMEM 时，编译器需要对齐和可整除性两个条件才能选择向量化指令。
a_tensor_ = (
    from_dlpack(a, assumed_align=32)        # 32-byte aligned base address
    .mark_layout_dynamic(leading_dim=1)
    .mark_compact_shape_dynamic(mode=1, divisibility=k)  # K dim divisible by k
)
b_tensor_ = (
    from_dlpack(b, assumed_align=32)
    .mark_layout_dynamic(leading_dim=1)
    .mark_compact_shape_dynamic(mode=1, divisibility=k)
)
c_tensor_ = (
    from_dlpack(c, assumed_align=32)
    .mark_layout_dynamic(leading_dim=1)
    .mark_compact_shape_dynamic(mode=1, divisibility=n)  # N dim divisible by n
)

In [ ]:
# 编译 kernel
vectorized_kernel = cute.compile(host_function, a_tensor_, b_tensor_, c_tensor_, kernel_with_prefetch)

# 运行 kernel
benchmark(vectorized_kernel, a_tensor_, b_tensor_, c_tensor_)

# 验证结果
ref = (torch.einsum("mk,nk->mn", a.to(torch.float32), b.to(torch.float32))).cpu()
torch.testing.assert_close(
    c.cpu(), ref.to(cutlass_torch.dtype(io_dtype)), atol=1e-1, rtol=1e-05
)
print("验证通过!")

性能指标:
-------------------
Kernel 执行时间: 781.8080 us
计算吞吐量: 1406.37 tflops
验证通过!
